# Phase 4 & 4b: Uncertainty Quantification & Prediction Intervals

**Project:** Agentic Predictive Maintenance  
**Dataset:** NASA C-MAPSS FD001  
**Objective:** Build and evaluate robust RUL uncertainty quantification methods:
1. **Heteroscedastic Binned-Residual Intervals (Default, Phase 4b):** Sharp, degradation-dependent intervals fitted empirically on validation residuals.
2. **Monte Carlo Dropout (Baseline Comparison, Phase 4):** Epistemic uncertainty estimation via stochastic forward passes.

---
### Key Findings:
* **MC-Dropout Inflation Issue:** MC-Dropout std had no useful correlation with prediction error (Spearman $\rho = -0.13$ on test, $-0.30$ on val), requiring a large uniform multiplier ($k=7.23$) that resulted in an excessively wide interval (~59.1 cycles) even for critical engines near failure.
* **Binned-Residual Superiority:** Binned residual intervals achieve **92.0% coverage** on the 100 test engines with an overall mean width of **36.5 cycles**, and narrow down to **11.6 cycles (91.7% coverage)** for engines with predicted RUL $< 30$ cycles.

In [1]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Add project root to path
PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "predictive-maintenance" else Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_cmapss
from src.preprocessing import (
    drop_uninformative_columns,
    compute_rul,
    scale_features,
    generate_sequences,
    get_test_sequences,
    RETAINED_SENSOR_COLUMNS,
)
from src.uncertainty import (
    predict_rul_with_interval,
    predict_point,
    fit_binned_residual_calibration,
    binned_residual_interval,
    mc_dropout_predict,
    interval_from_samples,
    compute_coverage,
    load_uncertainty_model_and_calib,
)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 120
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | PyTorch: {torch.__version__}")

Device: cpu | PyTorch: 2.13.0+cpu


## 1. Load Data & Winning Model Checkpoint

In [2]:
train_df, test_df, rul_df = load_cmapss("FD001", data_dir=PROJECT_ROOT / "data" / "raw")
train_clean = drop_uninformative_columns(train_df)
test_clean = drop_uninformative_columns(test_df)
train_rul = compute_rul(train_clean, max_rul=125, is_train=True)
train_scaled, test_scaled, _ = scale_features(train_rul, test_clean)

rng = np.random.default_rng(42)
unique_units = np.sort(train_scaled["unit_number"].unique())
val_units = np.sort(rng.permutation(unique_units)[:20])
val_df = train_scaled[train_scaled["unit_number"].isin(val_units)].copy()

X_val_all, y_val_all = generate_sequences(val_df, sequence_length=30)
X_test, test_unit_order = get_test_sequences(test_scaled, sequence_length=30)
y_test_raw = rul_df["RUL"].values.astype(np.float32)
y_test_capped = np.minimum(y_test_raw, 125.0)

model, _, target_scale = load_uncertainty_model_and_calib(
    checkpoint_path=PROJECT_ROOT / "outputs" / "best_lstm_model.pt", device=device
)
print(f"Loaded model: hidden_size={model.hidden_size}, dropout={model.dropout_rate}, target_scale={target_scale}")

Loaded model: hidden_size=64, dropout=0.2, target_scale=125.0


## 2. Fit Heteroscedastic Binned-Residual Calibration on Validation Data

In [3]:
y_pred_val = predict_point(model, X_val_all, target_scale=target_scale, device=device)
res_calib = fit_binned_residual_calibration(y_val_all, y_pred_val, alpha=0.10)
print(f"Validation Empirical Coverage across 3,272 windows: {res_calib['validation_empirical_coverage']*100:.1f}%")
for b in res_calib['bins']:
    print(f"  -> Bin [{b['edge_lower']:.0f}, {b['edge_upper']:.0f}) cyc: N={b['count']}, q05={b['q05']:+.2f}, q95={b['q95']:+.2f}, width={b['raw_width']:.2f} cyc")

Validation Empirical Coverage across 3,272 windows: 89.8%
  -> Bin [0, 25) cyc: N=516, q05=-2.95, q95=+6.53, width=9.48 cyc
  -> Bin [25, 50) cyc: N=456, q05=-9.05, q95=+13.09, width=22.14 cyc
  -> Bin [50, 75) cyc: N=425, q05=-21.16, q95=+21.72, width=42.88 cyc
  -> Bin [75, 100) cyc: N=473, q05=-30.16, q95=+32.02, width=62.18 cyc
  -> Bin [100, 125) cyc: N=1402, q05=-30.44, q95=+13.53, width=43.97 cyc


## 3. Test Set Evaluation & Side-by-Side Comparison

In [4]:
y_pred_test = predict_point(model, X_test, target_scale=target_scale, device=device)
low_res, up_res = binned_residual_interval(y_pred_test, calib_data=res_calib)
cov_res = compute_coverage(y_test_capped, low_res, up_res)
width_res = np.mean(up_res - low_res)

print(f"Binned Residual Test 90% Coverage: {cov_res*100:.1f}% (Mean width: {width_res:.2f} cycles)")

# Critical engines (< 30 cycles)
crit_mask = (y_pred_test < 30.0)
print(f"Safety-Critical Engines (Pred RUL < 30): Count={np.sum(crit_mask)}, Coverage={np.mean((y_test_capped[crit_mask] >= low_res[crit_mask]) & (y_test_capped[crit_mask] <= up_res[crit_mask]))*100:.1f}%, Mean Width={np.mean(up_res[crit_mask] - low_res[crit_mask]):.2f} cycles")

Binned Residual Test 90% Coverage: 92.0% (Mean width: 36.48 cycles)
Safety-Critical Engines (Pred RUL < 30): Count=24, Coverage=91.7%, Mean Width=11.59 cycles


## 4. Single-Sequence Inference via Unified Agent API (`predict_rul_with_interval`)

In [5]:
sample_window = X_test[0]
mean_val, low_val, up_val = predict_rul_with_interval(sample_window, method="binned_residual")
print(f"Engine #1 Default RUL Prediction: {mean_val:.2f} cycles [90% Interval: {low_val:.2f} - {up_val:.2f} cycles]")

Engine #1 Default RUL Prediction: 108.75 cycles [90% Interval: 78.31 - 122.28 cycles]
